In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.kernel_approximation import Nystroem
from sklearn.svm import LinearSVC
from scipy import sparse

In [2]:
data_folder = '/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof'

In [3]:
train = pd.read_csv(f"{data_folder}/train.csv")
test  = pd.read_csv(f"{data_folder}/test.csv")
sub   = pd.read_csv(f"{data_folder}/sample_submission.csv")

print(train.shape)
print(test.shape)
print(sub.shape)
train.head(2)

(3360, 6010)
(1000, 6002)
(1000, 9)


,sample_id,species_id,maldi_feature_0,maldi_feature_1,maldi_feature_2,maldi_feature_3,maldi_feature_4,maldi_feature_5,maldi_feature_6,maldi_feature_7,...,maldi_feature_5998,maldi_feature_5999,Ampicillin,Levofloxacin,Ciprofloxacin,Imipenem,Amoxicillin_Clavulanic_acid,Ertapenem,Cefotaxime,Cefuroxime
0,SAMPLE_00000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,SAMPLE_00001,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [12]:
# ============================
# TUNE ONLY Ampicillin
# ============================
TARGET_TUNE = "Ampicillin"

def run_cfg(gamma, n_components, C, quiet=True):
    # quiet=True: don’t print fold AUCs for every config
    # We temporarily silence per-fold prints by reusing the function but controlling prints.
    # Easiest: just comment out fold prints in cv_nystrom_linear_svm_auc while tuning.
    oof_auc, mean_auc, std_auc = cv_nystrom_linear_svm_auc(
        TARGET_TUNE, n_components=n_components, gamma=gamma, C=C
    )
    return {"gamma": gamma, "n_components": n_components, "C": C,
            "oof_auc": oof_auc, "fold_mean": mean_auc, "fold_std": std_auc}

# ---- Stage 0: baseline config (your current)
baseline = run_cfg(gamma=NY_GAMMA, n_components=NY_N_COMPONENTS, C=SVM_C)
print("\nBaseline config:", baseline)

# ---- Stage 1: gamma sweep (most important)
gamma_grid = [1e-6, 3e-6, 1e-5]
stage1 = []
for g in gamma_grid:
    print(f"\n[Stage 1] gamma={g}")
    stage1.append(run_cfg(gamma=g, n_components=NY_N_COMPONENTS, C=SVM_C))

df1 = pd.DataFrame(stage1).sort_values("oof_auc", ascending=False)
print("\n[Stage 1 results]")
print(df1.to_string(index=False))

best_g = float(df1.iloc[0]["gamma"])
print("\nBest gamma:", best_g)

# ---- Stage 2: n_components sweep
components_grid = [2200, 2400, 2600, 2800, 3000]
stage2 = []
for nc in components_grid:
    print(f"\n[Stage 2] n_components={nc}")
    stage2.append(run_cfg(gamma=best_g, n_components=nc, C=SVM_C))

df2 = pd.DataFrame(stage2).sort_values("oof_auc", ascending=False)
print("\n[Stage 2 results]")
print(df2.to_string(index=False))

best_nc = int(df2.iloc[0]["n_components"])
print("\nBest n_components:", best_nc)

# ---- Stage 3: C sweep
C_grid = [0.25]
stage3 = []
for c in C_grid:
    print(f"\n[Stage 3] C={c}")
    stage3.append(run_cfg(gamma=best_g, n_components=best_nc, C=c))

df3 = pd.DataFrame(stage3).sort_values("oof_auc", ascending=False)
print("\n[Stage 3 results]")
print(df3.to_string(index=False))

best = df3.iloc[0].to_dict()
print("\nBEST CONFIG FOUND:", best)

# ---- Keep only if it’s a real gain vs baseline
delta = best["oof_auc"] - baseline["oof_auc"]
print(f"\nDelta vs baseline: {delta:+.5f}")

if delta >= 0.003:
    print("KEEP tuned config (>= +0.003 OOF AUC).")
else:
    print("DISCARD tuning (gain too small); keep baseline config.")

[Ampicillin] Fold 1 AUC: 0.92196
[Ampicillin] Fold 2 AUC: 0.92724
[Ampicillin] Fold 3 AUC: 0.93192
[Ampicillin] Fold 4 AUC: 0.93341
[Ampicillin] Fold 5 AUC: 0.91045
[Ampicillin] OOF AUC: 0.92090 | mean±std: 0.92500 ± 0.00830

Baseline config: {'gamma': 0.01, 'n_components': 1200, 'C': 1.0, 'oof_auc': 0.9209016804138317, 'fold_mean': 0.9249962679571888, 'fold_std': 0.008299252032456506}

[Stage 1] gamma=1e-06
[Ampicillin] Fold 1 AUC: 0.92429
[Ampicillin] Fold 2 AUC: 0.92556
[Ampicillin] Fold 3 AUC: 0.94220
[Ampicillin] Fold 4 AUC: 0.94111
[Ampicillin] Fold 5 AUC: 0.91851
[Ampicillin] OOF AUC: 0.92555 | mean±std: 0.93033 ± 0.00955

[Stage 1] gamma=3e-06
[Ampicillin] Fold 1 AUC: 0.92592
[Ampicillin] Fold 2 AUC: 0.92610
[Ampicillin] Fold 3 AUC: 0.94133
[Ampicillin] Fold 4 AUC: 0.94255
[Ampicillin] Fold 5 AUC: 0.92023
[Ampicillin] OOF AUC: 0.92871 | mean±std: 0.93123 ± 0.00901

[Stage 1] gamma=1e-05
[Ampicillin] Fold 1 AUC: 0.92810
[Ampicillin] Fold 2 AUC: 0.92691
[Ampicillin] Fold 3 AUC: 0

/usr/local/lib/python3.11/dist-packages/sklearn/kernel_approximation.py:1001: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(


[Ampicillin] Fold 1 AUC: 0.92920


/usr/local/lib/python3.11/dist-packages/sklearn/kernel_approximation.py:1001: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(


[Ampicillin] Fold 2 AUC: 0.92691


...

[Ampicillin] Fold 3 AUC: 0.94283


/usr/local/lib/python3.11/dist-packages/sklearn/kernel_approximation.py:1001: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(


[Ampicillin] Fold 4 AUC: 0.94201


/usr/local/lib/python3.11/dist-packages/sklearn/kernel_approximation.py:1001: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(


[Ampicillin] Fold 5 AUC: 0.92164
[Ampicillin] OOF AUC: 0.93161 | mean±std: 0.93252 ± 0.00845

[Stage 2 results]
  gamma  n_components   C  oof_auc  fold_mean  fold_std
0.00001          2600 1.0 0.931735   0.932604  0.008428
0.00001          2800 1.0 0.931605   0.932518  0.008455
0.00001          3000 1.0 0.931605   0.932518  0.008455
0.00001          2400 1.0 0.931496   0.932339  0.007971
0.00001          2200 1.0 0.931474   0.932386  0.007918

Best n_components: 2600

[Stage 3] C=0.25
[Ampicillin] Fold 1 AUC: 0.92633
[Ampicillin] Fold 2 AUC: 0.92650
[Ampicillin] Fold 3 AUC: 0.94216
[Ampicillin] Fold 4 AUC: 0.94326
[Ampicillin] Fold 5 AUC: 0.91892
[Ampicillin] OOF AUC: 0.92880 | mean±std: 0.93144 ± 0.00961

[Stage 3 results]
  gamma  n_components    C  oof_auc  fold_mean  fold_std
0.00001          2600 0.25 0.928802   0.931435  0.009611

BEST CONFIG FOUND: {'gamma': 1e-05, 'n_components': 2600.0, 'C': 0.25, 'oof_auc': 0.9288018369483713, 'fold_mean': 0.9314350280530569, 'fold_std': 0.0

In [4]:
# ----------------------------
# Config
# ----------------------------
SEED = 42
N_SPLITS = 5

TARGETS_TO_TEST = [
    "Ampicillin"
]

SPECIES_COL = "species_id"
maldi_cols = [c for c in train.columns if c.startswith("maldi_feature_")]
assert len(maldi_cols) > 0, "No MALDI columns found with prefix 'maldi_feature_'."

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Nyström/SVM hyperparams (TUNED FOR AMPICILLIN)
NY_N_COMPONENTS = 2600     
NY_GAMMA = 1e-5            
SVM_C = 0.25             


def cv_nystrom_linear_svm_auc(target: str,
                             n_components=NY_N_COMPONENTS,
                             gamma=NY_GAMMA,
                             C=SVM_C):
    labeled = train.dropna(subset=[target]).copy()

    X_maldi = labeled[maldi_cols].values.astype(np.float32)
    X_species = labeled[[SPECIES_COL]].astype(str).values  # one-hot expects strings robustly
    y = labeled[target].astype(int).values

    oof_scores = np.zeros(len(labeled), dtype=np.float64)
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_maldi, y), start=1):
        Xm_tr, Xm_va = X_maldi[tr_idx], X_maldi[va_idx]
        Xs_tr, Xs_va = X_species[tr_idx], X_species[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        # 1) Scale MALDI (fit on train-fold only)
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xm_tr_s = scaler.fit_transform(Xm_tr)
        Xm_va_s = scaler.transform(Xm_va)

        # 2) Nyström feature map on MALDI (fit on train-fold only)
        nyst = Nystroem(kernel="rbf", gamma=gamma, n_components=n_components, random_state=SEED + fold)
        Z_tr = nyst.fit_transform(Xm_tr_s)   # shape: (n_tr, n_components)
        Z_va = nyst.transform(Xm_va_s)

        # 3) One-hot species (fit on train-fold only)
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
        S_tr = ohe.fit_transform(Xs_tr)      # sparse
        S_va = ohe.transform(Xs_va)

        # 4) Combine Nyström(MALDI) + OneHot(species)
        # Nyström output is dense; convert to sparse for efficient hstack
        Z_tr_sp = sparse.csr_matrix(Z_tr)
        Z_va_sp = sparse.csr_matrix(Z_va)

        X_tr_final = sparse.hstack([Z_tr_sp, S_tr], format="csr")
        X_va_final = sparse.hstack([Z_va_sp, S_va], format="csr")

        # 5) Linear SVM
        clf = LinearSVC(C=C, random_state=SEED + fold, max_iter=20000)
        clf.fit(X_tr_final, y_tr)

        # AUC can use decision_function directly
        va_score = clf.decision_function(X_va_final)
        oof_scores[va_idx] = va_score

        fold_auc = roc_auc_score(y_va, va_score)
        fold_aucs.append(fold_auc)
        print(f"[{target}] Fold {fold} AUC: {fold_auc:.5f}")

    oof_auc = roc_auc_score(y, oof_scores)
    print(f"[{target}] OOF AUC: {oof_auc:.5f} | mean±std: {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f}")
    return float(oof_auc), float(np.mean(fold_aucs)), float(np.std(fold_aucs))


# ----------------------------
# Run
# ----------------------------
results = []
for t in TARGETS_TO_TEST:
    print("\n==============================")
    print(f"Target: {t}")
    oof_auc, mean_auc, std_auc = cv_nystrom_linear_svm_auc(
        t, n_components=NY_N_COMPONENTS, gamma=NY_GAMMA, C=SVM_C
    )
    results.append({"target": t, "oof_auc": oof_auc, "fold_mean": mean_auc, "fold_std": std_auc})

res_df = pd.DataFrame(results).sort_values("oof_auc", ascending=False)
print("\n=== Nyström + LinearSVC summary ===")
print(res_df.to_string(index=False))



Target: Ampicillin
[Ampicillin] Fold 1 AUC: 0.92627
[Ampicillin] Fold 2 AUC: 0.92646
[Ampicillin] Fold 3 AUC: 0.94211
[Ampicillin] Fold 4 AUC: 0.94333
[Ampicillin] Fold 5 AUC: 0.91892
[Ampicillin] OOF AUC: 0.92883 | mean±std: 0.93142 ± 0.00963

=== Nyström + LinearSVC summary ===
    target  oof_auc  fold_mean  fold_std
Ampicillin 0.928827   0.931418  0.009629


In [9]:
# labeled_df is the dataframe you used for Ampicillin (train rows where label is not NaN)
oof_out = pd.DataFrame({
    "sample_id": labeled_df["sample_id"].values,
    "ampicillin_svm_oof": oof_scores
})
oof_out.to_csv("/kaggle/working/oof_ampicillin_nystrom_svm.csv", index=False)
print("Wrote: oof_ampicillin_nystrom_svm.csv")


NameError: name 'oof_scores' is not defined

In [7]:
# ---- Fit final Nyström+SVM on ALL labeled Ampicillin and predict test ----
# (uses the SAME hyperparams you selected)

TARGET = "Ampicillin"

labeled_df = train.dropna(subset=[TARGET]).copy()
X_maldi_lab = labeled_df[maldi_cols].values.astype(np.float32)
X_species_lab = labeled_df[[SPECIES_COL]].astype(str).values
y_lab = labeled_df[TARGET].astype(int).values

# Test features
X_maldi_test = test[maldi_cols].values.astype(np.float32)
X_species_test = test[[SPECIES_COL]].astype(str).values

# 1) Scale MALDI (fit on all labeled)
scaler = StandardScaler(with_mean=True, with_std=True)
Xm_lab_s = scaler.fit_transform(X_maldi_lab)
Xm_test_s = scaler.transform(X_maldi_test)

# 2) Nyström mapping (fit on all labeled)
nyst = Nystroem(kernel="rbf", gamma=NY_GAMMA, n_components=NY_N_COMPONENTS, random_state=SEED)
Z_lab = nyst.fit_transform(Xm_lab_s)
Z_test = nyst.transform(Xm_test_s)

# 3) One-hot species (fit on all labeled species)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
S_lab = ohe.fit_transform(X_species_lab)
S_test = ohe.transform(X_species_test)

# 4) Combine
Z_lab_sp = sparse.csr_matrix(Z_lab)
Z_test_sp = sparse.csr_matrix(Z_test)
X_lab_final = sparse.hstack([Z_lab_sp, S_lab], format="csr")
X_test_final = sparse.hstack([Z_test_sp, S_test], format="csr")

# 5) Train final LinearSVC and predict test decision scores
clf = LinearSVC(C=SVM_C, random_state=SEED, max_iter=20000)
clf.fit(X_lab_final, y_lab)

test_scores = clf.decision_function(X_test_final)

test_out = pd.DataFrame({
    "sample_id": test["sample_id"].values,
    "ampicillin_svm_test": test_scores
})
test_out.to_csv("test_ampicillin_nystrom_svm.csv", index=False)
print("Wrote: test_ampicillin_nystrom_svm.csv")


Wrote: test_ampicillin_nystrom_svm.csv


In [ ]:
oof = pd.read_csv("oof_ampicillin_nystrom_svm.csv")
te  = pd.read_csv("test_ampicillin_nystrom_svm.csv")

assert oof["sample_id"].is_unique
assert te["sample_id"].is_unique
assert oof.shape[1] == 2 and te.shape[1] == 2
assert np.isfinite(oof["ampicillin_svm_oof"]).all()
assert np.isfinite(te["ampicillin_svm_test"]).all()
